In [0]:
%sql
-- ============================================================
-- Restockify Workflow — Owned-Schema Bootstrap
-- ============================================================
-- Data Engineering now owns the real inventory/consumption/procurement/
-- restock-request data as a star schema in gold_dev.dim +
-- gold_dev.supply_chain_analytics (see src/agentic_restock/config.py and
-- docs/architecture.md §6 for the full mapping). The 5 flat mock tables
-- this notebook used to create/seed here are retired.
--
-- ab_training.agentic_restock now houses only the artifacts we own outright:
--   1. The §4.2 Unity Catalog functions (deployed separately, see
--      notebooks/uc_functions/deep_analysis_functions.ipynb).
--   2. quote_metadata — a thin companion table for the Teams/Review-App
--      fields (summary_report, teams_message_id, databricks_preview_url,
--      decision_comments, ...) that have no home in Data Engineering's
--      fact_restock_request, because that fact's grain is one row per
--      requested part-line per quote, not one row per quote header.
-- ============================================================

CREATE SCHEMA IF NOT EXISTS ab_training.agentic_restock
COMMENT 'Restockify Workflow artifacts we own outright: the section 4.2 Unity Catalog functions and quote_metadata (Teams/Review-App fields keyed by quote_id). Inventory/consumption/procurement/restock-request data itself lives in Data Engineering''s gold_dev star schema (gold_dev.dim, gold_dev.supply_chain_analytics).';

In [0]:
%sql
-- ============================================================
-- Table: quote_metadata
-- Companion table to Data Engineering's gold_dev.supply_chain_analytics.
-- fact_restock_request. That fact is grain-per-part-line-per-quote, so
-- there's no natural home there for quote-header fields (one per QUOTE_ID,
-- not one per line). This table fills that gap. Joined by the Databricks
-- Review App and the Supervisor Agent on quote_id ->
-- gold_dev.supply_chain_analytics.fact_restock_request.QUOTE_ID.
-- Status/urgency/decision themselves are NOT duplicated here — they live in
-- gold_dev.dim.dim_request_status (via fact_restock_request.REQUEST_STATUS_KEY),
-- read fresh on every join to avoid staleness.
-- ============================================================

CREATE OR REPLACE TABLE ab_training.agentic_restock.quote_metadata (
  quote_id STRING NOT NULL COMMENT 'Business key matching gold_dev.supply_chain_analytics.fact_restock_request.QUOTE_ID',
  summary_report STRING COMMENT 'Genie Agent natural-language assumption/reasoning report for this quote',
  teams_message_id STRING COMMENT 'Reference ID of the Adaptive Card sent to Teams',
  teams_sent_at TIMESTAMP COMMENT 'When the Teams notification was dispatched',
  databricks_preview_url STRING COMMENT 'Deep link to the Databricks Review App for this quote',
  decision_comments STRING COMMENT 'Optional approver comments explaining the decision',
  created_by STRING COMMENT 'Agent that created this quote record (default: supervisor_agent)',
  created_at TIMESTAMP COMMENT 'Row creation timestamp',
  updated_at TIMESTAMP COMMENT 'Last modification timestamp',
  CONSTRAINT pk_quote_metadata PRIMARY KEY (quote_id)
)
COMMENT 'Teams/Review-App metadata per quote header, keyed by quote_id. Companion to gold_dev.supply_chain_analytics.fact_restock_request (which is grain-per-part-line, not per quote) -- join on quote_id = QUOTE_ID for the full picture.';

-- Seed rows for 3 of the QUOTE_IDs Data Engineering already seeded in
-- gold_dev.supply_chain_analytics.fact_restock_request, covering the three
-- interesting lifecycle states (PENDING_APPROVAL, REJECTED, COMPLETED) so the
-- Review App and Genie Agent have realistic metadata to join against.
INSERT INTO ab_training.agentic_restock.quote_metadata VALUES
  (
    'QT-2026-0001',
    'PENDING_APPROVAL: two part-lines flagged, one CRITICAL and one HIGH urgency. Awaiting PM review in the Databricks Review App.',
    'msg-teams-0001',
    '2026-08-17 09:00:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0001',
    NULL,
    'supervisor_agent',
    '2026-08-17 08:58:00',
    '2026-08-17 09:00:00'
  ),
  (
    'QT-2026-0006',
    'REJECTED: PM decided the CRITICAL and HIGH urgency lines did not warrant restocking this cycle.',
    'msg-teams-0006',
    '2026-08-17 07:30:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0006',
    'Rejected -- alternate supplier already covering this shortfall outside the system.',
    'supervisor_agent',
    '2026-08-17 07:25:00',
    '2026-08-17 09:15:00'
  ),
  (
    'QT-2026-0010',
    'COMPLETED: both part-lines approved and fulfilled; Restock Agent confirmed real-time stock and closed out the quote.',
    'msg-teams-0010',
    '2026-08-17 06:45:00',
    'https://workspace.databricks.com/apps/restock-review?quote_id=QT-2026-0010',
    'Approved -- critical for production line continuity.',
    'supervisor_agent',
    '2026-08-17 06:40:00',
    '2026-08-17 10:20:00'
  );

In [0]:
%sql
-- ============================================================
-- RETIRED: threshold_config_table
-- Folded into Data Engineering's gold_dev.supply_chain_analytics.
-- fact_inventory_snapshot (SAFETY_STOCK_QTY = reorder trigger,
-- MAX_STOCK_LEVEL = restock target). No mock table is created here
-- anymore -- see git history before this migration for the original
-- mock CREATE TABLE / INSERT statements.
-- ============================================================
SELECT 'threshold_config_table retired -- see fact_inventory_snapshot in gold_dev.supply_chain_analytics' AS note;

In [0]:
%sql
-- ============================================================
-- RETIRED: consumption_history
-- Replaced by Data Engineering's gold_dev.supply_chain_analytics.
-- fact_inventory_transaction (TRANSACTION_TYPE = 'ISSUE' rows are the
-- consumption events; see avg_daily_consumption() in
-- notebooks/uc_functions/deep_analysis_functions.ipynb). No mock table is
-- created here anymore -- see git history before this migration for the
-- original mock CREATE TABLE / INSERT statements.
-- ============================================================
SELECT 'consumption_history retired -- see fact_inventory_transaction in gold_dev.supply_chain_analytics' AS note;

In [0]:
%sql
-- ============================================================
-- RETIRED: open_request
-- Replaced by Data Engineering's gold_dev.supply_chain_analytics.
-- fact_restock_request (joined to gold_dev.dim.dim_request_status for
-- status/urgency/decision) plus our own quote_metadata table (cell 1
-- above) for the Teams/Review-App header fields that fact doesn't carry.
-- No mock table is created here anymore -- see git history before this
-- migration for the original mock CREATE TABLE / INSERT statements.
-- ============================================================
SELECT 'open_request retired -- see fact_restock_request in gold_dev.supply_chain_analytics + quote_metadata' AS note;

In [0]:
%sql
-- ============================================================
-- RETIRED: restock_requests
-- Merged into Data Engineering's gold_dev.supply_chain_analytics.
-- fact_restock_request, which is now one accumulating-snapshot fact
-- spanning both the old open_request lifecycle AND the old
-- restock_requests fulfillment ledger (RESTOCK_REQUEST_ID,
-- CONFIRMED_QTY, FULFILLED_DATE_KEY columns cover what restock_requests
-- used to). No mock table is created here anymore -- see git history
-- before this migration for the original mock CREATE TABLE / INSERT
-- statements.
-- ============================================================
SELECT 'restock_requests retired -- merged into fact_restock_request in gold_dev.supply_chain_analytics' AS note;

In [0]:
%sql
-- ============================================================
-- Verification: quote_metadata row count + a live join sanity check
-- against Data Engineering's gold_dev fact_restock_request, proving the
-- quote_id linkage actually resolves for the 3 seeded example quotes.
-- ============================================================

SELECT 'quote_metadata' AS table_name, COUNT(*) AS row_count FROM ab_training.agentic_restock.quote_metadata;

In [0]:
%sql
-- ============================================================
-- Verification (continued): quote_metadata joined live against
-- gold_dev.supply_chain_analytics.fact_restock_request +
-- gold_dev.dim.dim_request_status, confirming the quote_id linkage
-- resolves and the two sources compose into one coherent quote view.
--
-- The §4.1 Lakeflow Job coarse-check query itself now lives in
-- src/agentic_restock/jobs/lakeflow_trigger.py::build_coarse_check_query()
-- (it's generated in Python, not hand-copied here, so this notebook can't
-- drift out of sync with what the job actually runs).
-- ============================================================

SELECT
  qm.quote_id,
  drs.REQUEST_STATUS AS request_status,
  drs.URGENCY_LEVEL AS urgency_level,
  drs.DECISION AS decision,
  qm.summary_report,
  qm.databricks_preview_url,
  COUNT(*) AS n_part_lines
FROM ab_training.agentic_restock.quote_metadata qm
JOIN gold_dev.supply_chain_analytics.fact_restock_request frr ON frr.QUOTE_ID = qm.quote_id
JOIN gold_dev.dim.dim_request_status drs ON drs.REQUEST_STATUS_KEY = frr.REQUEST_STATUS_KEY
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY qm.quote_id;